# Deep Learning for BCI

> A serious attempt at all the assignments is mandatory to grant access to the final exam. Refer to the course manual for more details (section *Overview* on Brightspace). 

> Please add today's topic to your knowledge graph.

**Learning goals:**
- Get familiar with the EEGNet architecture and its implementation;
- Rehearsing concepts of regularization in the context of CNNs;
- Rehearsing convolutions in 2D and 1D;
- Implementing visualisations of the filters learned by EEGNet in the context of EEG filters and patterns;

**Solutions file:**
For this assignment, a solutions file will be released after the deadline.

**Note:**
In providing your answers to the questions, please denote which question you are providing an answer to (either in text or code) by commenting it with something like "Exercise 1.1" and "Exercise 3.4". This improves readability of your solutions as well as the quality of the feedback you will get.


## Top-down view onto this notebook
In this notebook you will:
- work through a provided pytorch implemenation of the EEGNet defined in the [Braindecode].(https://braindecode.org/stable/index.html) library (version 0.7). For this you will be provided with a set of relevant functions and classed, which are the building blocks for the pytorch implementation.
- dive into the concept of normalisation and its application along the different components of EEGNet.
- revisit the concept of convolution and discuss its application in individual layers of the EEGNet.
- combine an EEGNet based decoder with data from the MOABB library to build a full decoding pipeline including cross validation.
- visualize the trained model and link its components to filters in the time and frequency domain.
- work out differences in the pytorch and the original tensorflow based implementation of the original [EEGNet](http://arxiv.org/abs/1611.08024) authors.
- modify the pytorch implementation according to the identified differences. This especially includes an implementation of an early stopping procedure for the training of the CNN.

Like in assignment 12, we will again make use of the MOABB library.
Additionally, we will use the [pytorch](https://pytorch.org/get-started/locally/) library.



In [2]:
import math
import time
from copy import deepcopy

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import mne

from moabb.paradigms import LeftRightImagery, P300
from moabb.datasets import BNCI2014_001, BNCI2014_009
from moabb.evaluations import WithinSessionEvaluation

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

from scipy import signal

import warnings
warnings.filterwarnings('ignore', message='.*warnEpochs', )   # reduce noise from moabb
mne.set_log_level('WARNING')

np.random.seed(42)

---

## Exercise 1: Get familiar with the code 
In this assignment, we will be interested in deep learning approaches for BCI data classification. 
In particular, we will take a close look at Lawhern et al. (2018) (https://doi.org/10.1088/1741-2552/aace8c, also available here: http://arxiv.org/abs/1611.08024)

In this first exercise, you will have to get familiar with the code we provided you. 
This code consists of: a small function that loads MOABB data, a pytorch implementation of EEGNet, 
a function that trains pytorch neural networks, and a set of functions to plot EEGNet's filters.
First, read the provided code, and only then, carefully answer the questions below to make sure you understood the code. 

<a id='exercise1_questions'></a>
### Questions 
1. What is the sampling frequency of the data returned by the `get_moabb_data` function?
1. To train neural networks, regularisation methods are needed. Some have directly been built into the EEGNet architecture. Find which regularisations are implemented, name in which layer they each can be found and briefly explain their expected effect.
1. In the code we provided, the **temporal filtering** is implemented as a 2D convolution (same goes for the spatial convolution). If we wanted to implement this spatial convolution with a **1D convolution** instead, what operation(s) should be applied on the data before that convolution to make it work? 
1. Now, we would like to re-implement the **spatial filtering**, currently also implemented as a 2D convolution, with a **linear layer** (i.e. a fully connected layer, or `torch.nn.Linear`), **in the case where `F1=1`**. What operation(s) should you apply to the data for this to work?
1. In the training function, we can observe that the loss function used for training is called "NLLLoss". Find in the pytorch documentation what this abreviation stands for and if this loss should be used for regression or classification problems.
1. What is the name of the optimizer used in the training function?
1. In the description of the function `get_EEGNet_spatial_filters` defined below, we can see that the filters are returned as an array of shape `(F1, D, in_chans)`. What do these numbers correspond to? Why are the filters returned as a 3D array and not a 2D one as usual?
1. **Full pipeline** - use all the functions and classes we provided you to execute the following steps.
    1. load the data from dataset BNCI2014_001, subject 1, `1test` (also referred to as `session_E` in older moabb versions);
    1. split the data you loaded to use the first 100 example as train set and the last 44 as validation set;
    1. instantiate `EEGNetv4` with the default hyper-parameters;
    1. train the network using the training and validation data you loaded for 50 epochs (i.e. 50 iterations, being passes through the whole dataset; not "BCI data epochs");
    1. extract the temporal filters of the trained network;
    1. extract the spatial filters of the trained network;
    1. finally, plot the extracted spatial and temporal filters.

[Take me to the answers section of exercise 1 -->](#exercise1_answers)

### Provided code


#### Loading MOABB data
In the cell below, we defined a small function that allows you to load data from the MOABB library with the correct pre-processing parameters for this implementation of EEGNet.  


In [ ]:
def get_moabb_data(paradigm_class, dataset, subject, session=None, tmin=0.0, tmax=None):
    # Pre-processing parameters:
    # We use the same pre-processing parameters as in the eperiments of the EEGNet paper 
    fmin = 1
    fmax = 40
    resample = 128

    # Load the data: 
    paradigm = paradigm_class(fmin=fmin, fmax=fmax, resample=resample, tmin=tmin, tmax=tmax)
    epochs, y, metadata = paradigm.get_data(dataset, subjects=[subject], return_epochs=True)

    if session is not None:
        # We only keep the data from one session:
        epochs, y = epochs[metadata.session==session], y[metadata.session==session]

    # We extract a numpy array and multiply by 1e6 to go from Volts to micro Volts
    X = epochs.get_data().astype('float32') * 1e6
    
    return X, y, epochs.info, metadata


#### Some building blocks of EEGNet
In this assignment, we use the pytorch implementation of EEGNet defined in the [Braindecode](https://braindecode.org/stable/index.html) library (version 0.7).
To implement EEGNet, they both used **standard building blocks** from the pytorch library (called modules), and some **custom ones** that they defined themselves.

You can find all the information you need on the **standard modules** in the [pytorch documentation](https://pytorch.org/docs/stable/nn.html),
and on the **custom ones** in the cell below.
We copied below the EEGNet implementation as well as all the custom blocks and functions needed that were defined in the Braindecode library so that you don't have to install it.  

In [ ]:

class Conv2dWithConstraint(nn.Conv2d):
    def __init__(self, *args, max_norm=1, **kwargs):
        self.max_norm = max_norm
        super(Conv2dWithConstraint, self).__init__(*args, **kwargs)

    def forward(self, x):
        self.weight.data = torch.renorm(
            self.weight.data, p=2, dim=0, maxnorm=self.max_norm
        )
        return super(Conv2dWithConstraint, self).forward(x)


class Ensure4d(nn.Module):
    def forward(self, x):
        while len(x.shape) < 4:
            x = x.unsqueeze(-1)
        return x


class Expression(nn.Module):
    """Compute given expression on forward pass.
    Parameters
    ----------
    expression_fn : callable
        Should accept variable number of objects of type
        `torch.autograd.Variable` to compute its output.
    """

    def __init__(self, expression_fn):
        super(Expression, self).__init__()
        self.expression_fn = expression_fn

    def forward(self, *x):
        return self.expression_fn(*x)

    def __repr__(self):
        if hasattr(self.expression_fn, "func") and hasattr(
            self.expression_fn, "kwargs"
        ):
            expression_str = "{:s} {:s}".format(
                self.expression_fn.func.__name__, str(self.expression_fn.kwargs)
            )
        elif hasattr(self.expression_fn, "__name__"):
            expression_str = self.expression_fn.__name__
        else:
            expression_str = repr(self.expression_fn)
        return (
            self.__class__.__name__ +
            "(expression=%s) " % expression_str
        )

def squeeze_final_output(x):
    """Removes empty dimension at end and potentially removes empty time
     dimension. It does  not just use squeeze as we never want to remove
     first dimension.
    Returns
    -------
    x: torch.Tensor
        squeezed tensor
    """

    assert x.size()[3] == 1
    x = x[:, :, :, 0]
    if x.size()[2] == 1:
        x = x[:, :, 0]
    return x

def _transpose_to_b_1_c_0(x):
    return x.permute(0, 3, 1, 2)


def _transpose_1_0(x):
    return x.permute(0, 1, 3, 2)

def _glorot_weight_zero_bias(model):
    """Initalize parameters of all modules by initializing weights with
    glorot
     uniform/xavier initialization, and setting biases to zero. Weights from
     batch norm layers are set to 1.

    Parameters
    ----------
    model: Module
    """
    for module in model.modules():
        if hasattr(module, "weight"):
            if not ("BatchNorm" in module.__class__.__name__):
                nn.init.xavier_uniform_(module.weight, gain=1)
            else:
                nn.init.constant_(module.weight, 1)
        if hasattr(module, "bias"):
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)


#### An implementation of EEGNet in pytorch
In the cell below, you will find the pytorch implementation of EEGNet, copied from the [Braindecode](https://braindecode.org/stable/index.html) library (version 0.7).

In [ ]:
                
class EEGNetv4(nn.Sequential):
    """EEGNet v4 model from Lawhern et al 2018.

    See details in [EEGNet4]_.

    Parameters
    ----------
    in_chans : int
        XXX

    Notes
    -----
    This implementation is not guaranteed to be correct, has not been checked
    by original authors, only reimplemented from the paper description.

    References
    ----------
    .. [EEGNet4] Lawhern, V. J., Solon, A. J., Waytowich, N. R., Gordon,
       S. M., Hung, C. P., & Lance, B. J. (2018).
       EEGNet: A Compact Convolutional Network for EEG-based
       Brain-Computer Interfaces.
       arXiv preprint arXiv:1611.08024.
    """

    def __init__(
        self,
        in_chans,
        n_classes,
        input_window_samples=None,
        final_conv_length="auto",
        pool_mode="mean",
        F1=8,
        D=2,
        F2=16,  # usually set to F1*D (?)
        kernel_length=64,
        third_kernel_size=(8, 4),
        drop_prob=0.25,
    ):
        super().__init__()
        if final_conv_length == "auto":
            assert input_window_samples is not None
        self.in_chans = in_chans
        self.n_classes = n_classes
        self.input_window_samples = input_window_samples
        self.final_conv_length = final_conv_length
        self.pool_mode = pool_mode
        self.F1 = F1
        self.D = D
        self.F2 = F2
        self.kernel_length = kernel_length
        self.third_kernel_size = third_kernel_size
        self.drop_prob = drop_prob

        pool_class = dict(max=nn.MaxPool2d, mean=nn.AvgPool2d)[self.pool_mode]
        self.add_module("ensuredims", Ensure4d())
        # b c 0 1
        # now to b 1 0 c
        self.add_module("dimshuffle", Expression(_transpose_to_b_1_c_0))

        self.add_module(
            "conv_temporal",
            nn.Conv2d(
                1,
                self.F1,
                (1, self.kernel_length),
                stride=1,
                bias=False,
                padding=(0, self.kernel_length // 2),
            ),
        )
        self.add_module(
            "bnorm_temporal",
            nn.BatchNorm2d(self.F1, momentum=0.01, affine=True, eps=1e-3),
        )
        self.add_module(
            "conv_spatial",
            Conv2dWithConstraint(
                self.F1,
                self.F1 * self.D,
                (self.in_chans, 1),
                max_norm=1,
                stride=1,
                bias=False,
                groups=self.F1,
                padding=(0, 0),
            ),
        )

        self.add_module(
            "bnorm_1",
            nn.BatchNorm2d(
                self.F1 * self.D, momentum=0.01, affine=True, eps=1e-3
            ),
        )
        self.add_module("elu_1", Expression(F.elu))

        self.add_module("pool_1", pool_class(kernel_size=(1, 4), stride=(1, 4)))
        self.add_module("drop_1", nn.Dropout(p=self.drop_prob))

        # https://discuss.pytorch.org/t/how-to-modify-a-conv2d-to-depthwise-separable-convolution/15843/7
        self.add_module(
            "conv_separable_depth",
            nn.Conv2d(
                self.F1 * self.D,
                self.F1 * self.D,
                (1, 16),
                stride=1,
                bias=False,
                groups=self.F1 * self.D,
                padding=(0, 16 // 2),
            ),
        )
        self.add_module(
            "conv_separable_point",
            nn.Conv2d(
                self.F1 * self.D,
                self.F2,
                (1, 1),
                stride=1,
                bias=False,
                padding=(0, 0),
            ),
        )

        self.add_module(
            "bnorm_2",
            nn.BatchNorm2d(self.F2, momentum=0.01, affine=True, eps=1e-3),
        )
        self.add_module("elu_2", Expression(F.elu))
        self.add_module("pool_2", pool_class(kernel_size=(1, 8), stride=(1, 8)))
        self.add_module("drop_2", nn.Dropout(p=self.drop_prob))

        out = self(
            torch.ones(
                (1, self.in_chans, self.input_window_samples, 1),
                dtype=torch.float32
            )
        )
        n_out_virtual_chans = out.cpu().data.numpy().shape[2]

        if self.final_conv_length == "auto":
            n_out_time = out.cpu().data.numpy().shape[3]
            self.final_conv_length = n_out_time

        self.add_module(
            "conv_classifier",
            nn.Conv2d(
                self.F2,
                self.n_classes,
                (n_out_virtual_chans, self.final_conv_length),
                bias=True,
            ),
        )
        self.add_module("softmax", nn.Softmax(dim=1))  # the only line we changed from the braindecode implementation (to output probabilities)
        # Transpose back to the the logic of braindecode,
        # so time in third dimension (axis=2)
        self.add_module("permute_back", Expression(_transpose_1_0))
        self.add_module("squeeze", Expression(squeeze_final_output))

        _glorot_weight_zero_bias(self)
        
        
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


#### Training EEGNet
We defined in the cell below a function that trains a pytorch neural network on data represented as numpy arrays.

In [ ]:
def train_pytorch_net(module, X, y, X_val=None, y_val=None, n_epochs=500, batch_size=50, device='cpu', verbose=2):
    '''
    Trains a pytorch neural network with the ____ optimizer
    and the NLL loss.
    
    Parameters
    ----------
    module: torch.nn.Module 
        the pytorch neural network to train.
    X: numpy array, shape (n_train_examples, n_channels, n_time_samples), 
        the training data.
    y: numpy array, shape (n_train_examples,),
        the training labels.
    X_val: numpy array, shape (n_val_examples, n_channels, n_time_samples), 
        the optional validation data.
    y_val: numpy array, shape (n_val_examples,), 
        the optional validation labels.
    n_epochs: int, 
        the number of passes through the whole training dataset.
    batch_size: int, 
        the number of examples to use in every batch.
    device: str, 
        the device to use for training. 
        Can be: 'cpu' (default), 'cuda:0' for NVIDIA GPUs, or
        'mps' for MacBook Pros with M1/M2 chips. 
    verbose: int, 
        level of verbosity. 0: no log, 1: only last epoch, 2: at every epoch.
    '''
    module.to(device)
        
    # We convert the string labels into integers codes (for pytorch):
    le = LabelEncoder().fit(y)
    y_code = le.transform(y)
    
    # Transform the data into a pytorch-compatible format:
    dataset = TensorDataset(torch.tensor(X).to(device), torch.tensor(y_code).to(device))
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    if X_val is not None and y_val is not None:
        y_code_val = le.transform(y_val) 
        dataset_val = TensorDataset(torch.tensor(X_val).to(device), torch.tensor(y_code_val).to(device))
        dataloader_val = DataLoader(dataset_val, shuffle=False, batch_size=batch_size)
        losses_val = []
        accuracies_val = []
    else: 
        losses_val = None
        accuracies_val = None
    losses = []
    accuracies = []
    loss_fn = torch.nn.NLLLoss()
    optimizer = torch.optim.Adam(params=module.parameters())
    
    #Printing:
    if verbose>=1:
        str_epoch = 'epoch'
        digits_epoch = max(int(math.log10(n_epochs))+1, len(str_epoch))
        str_epoch = str_epoch.rjust(digits_epoch)
        message = f'{str_epoch}  duration  train_loss  train_acc'
        bar = '-'*len(str_epoch)+'  --------  ----------  ---------'
        if losses_val is not None:
            message += f'  val_loss  val_acc'
            bar += '  --------  -------'
        print(message)
        print(bar)
    for idx_epoch in range(1, n_epochs+1):
        t0 = time.time() 
        
        # Training:
        module.train()
        batch_losses = []
        correct, total = 0, 0
        for x, y_target in dataloader:
            optimizer.zero_grad()
            y_pred = module(x)
            loss = loss_fn(y_pred, y_target.long())
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.cpu().detach().item())
            correct += (y_pred.argmax(dim=1) == y_target).sum().cpu().detach().item()
            total += y_target.shape[0]
            del x, y_target, y_pred, loss
        losses.append(np.mean(batch_losses))
        accuracies.append(correct/total)

        # Validation:
        if losses_val is not None:
            module.eval()
            batch_losses = []
            correct, total = 0, 0
            with torch.no_grad():
                for x, y_target in dataloader_val:
                    y_pred = module(x)
                    loss = loss_fn(y_pred, y_target.long())
                    batch_losses.append(loss.cpu().detach().item())
                    correct += (y_pred.argmax(dim=1) == y_target).sum().cpu().detach().item()
                    total += y_target.shape[0]
                    del x, y_target, y_pred, loss
            losses_val.append(np.mean(batch_losses))
            accuracies_val.append(correct/total)
        
        # Printing:
        if verbose>=2 or (verbose>=1 and idx_epoch==n_epochs):
            message = f'{idx_epoch: {digits_epoch}}  {time.time()-t0:7.2f}s  {losses[-1]:10.4f}  {accuracies[-1]:9.4f}'
            if losses_val is not None:
                message += f'  {losses_val[-1]:8.4f}  {accuracies_val[-1]:7.4f}'
            print(message)
    return module, losses, accuracies, losses_val, accuracies_val


#### Plotting EEGNet's filters
Finally, we defined in the cell bellow three functions that can extract 
the convolutional kernels of EEGNet (i.e. the temporal and spatial filters), 
and plot them.

In [ ]:
def get_EEGNet_temporal_filters(module):
    ''' returns a numpy array of shape (F1, kernel_length) '''
    return module.get_submodule('conv_temporal').weight.squeeze(2).squeeze(1).cpu().detach().numpy()

def get_EEGNet_spatial_filters(module):
    ''' returns a numpy array of shape (F1, D, in_chans)'''
    return module.get_submodule('conv_spatial').weight.squeeze(3).squeeze(1).cpu().detach().numpy().reshape(module.F1, module.D, module.in_chans) 

def plot_EEGNet_filters(temporal_filters, spatial_filters, info, figsize=None, title=None, sfreq: int = 128):
    '''
    Plots the temporal filters of EEGNet and their corresponding spatial filters
    
    Parameters
    ----------
    temporal_filters: numpy array, shape (F1, kernel_length) 
        returned by get_EEGNet_temporal_filters
    spatial_filters: numpy array, shape (F1, D, in_chans)
        returned by get_EEGNet_spatial_filters
    info: mne.Info
        the info object corresponding to the original signal (also returned by get_MOABB_data) 
    figsize: (float, float)
        size of the figure
    title: str
        optional title of the figure
    '''
    F1, D, in_chans = spatial_filters.shape
    f1, kernel_length = temporal_filters.shape
    if not f1==F1:
        raise ValueError()
    fig = plt.figure(constrained_layout=True, figsize=figsize)
    if title is not None:
        fig.suptitle(title)
    fig_temp, fig_freq_res, fig_spat, fig_pat = fig.subfigures(4, 1, wspace=0.07)
    
    # Plot temporal filters:
    axes = fig_temp.subplots(1, F1, squeeze=False)
    t = np.arange(kernel_length)/info['sfreq']
    for i, (ax, f) in enumerate(zip(axes[0], temporal_filters)):
        ax.plot(t, f)
        ax.set_xlabel('time [s]')
        ax.set_title(f'Temp. filter {i+1}')
        ax.set_yticks([])
        
    # Plot the frequency response of the temporal filters
    axes = fig_freq_res.subplots(1, F1, squeeze=False)
    for i, (ax, f) in enumerate(zip(axes[0], temporal_filters)):
        f_norm = f / f.sum()
        freq_res = np.abs(np.fft.fft(f_norm))[0:f.shape[0]//2 + 1]
        freqs = np.linspace(0, sfreq // 2, freq_res.shape[0])
        
        ax.plot(freqs, freq_res)
        ax.set_xlabel('Frequency [Hz]')
        ax.set_title(f'Freq. response')
        ax.set_yticks([])
        
        if i == 0:
            ax.set_ylabel('Gain [dB]')
    
    # Plot spatial filters:
    axes = fig_spat.subplots(D, F1, squeeze=False)
    for i, (axx, ff) in enumerate(zip(axes.T, spatial_filters)):
        for j, (ax, f) in enumerate(zip(axx, ff)):
            mne.viz.plot_topomap(f, info, axes=ax, vlim=(-1,1), show=False, extrapolate='local')
            if i==0:
                ax.set_ylabel(f'Spat. filter {j+1}')

    # Plot spatial patterns:
    spatial_patterns = np.linalg.pinv(spatial_filters.reshape(F1*D, in_chans)).T.reshape(F1, D, in_chans)
    axes = fig_pat.subplots(D, F1, squeeze=False)
    for i, (axx, ff) in enumerate(zip(axes.T, spatial_patterns)):
        for j, (ax, f) in enumerate(zip(axx, ff)):
            mne.viz.plot_topomap(f, info, axes=ax, vlim=(-2,2), show=False, extrapolate='local')
            if i==0:
                ax.set_ylabel(f'Spat. pattern {j+1}')

    return fig, axes
    


<a id='exercise1_answers'></a>
### Your answers
[<-- Take me back to the questions section of exercise 1](#exercise1_questions)
> Answer here:

In [ ]:
# Or here:


---

## Exercise 2: Reproduce results from the article
As you might have noticed, the final validation accuracy we obtained in the previous exercise is not great, barely above chance level.
Similary, neither the filters or the patterns plotted look interesting.
When looking at the results presented in the article of Lawhern et al. (2018), 
we observe that they obtained a higher average classification accuracy in the within-subject case 
and much nicer filters in Figure 7.
However, the comparison is not fair because the conditions are not the same.

Our goal in this exercise will be to reproduce the plot in figure 7 of the article.


**Questions:**
1. In order to reproduce the results of Lawhern et al., we will first evaluate what are the differences between their pipeline and the one we defined in Exercise 1:
    1. list all the algorithmic differences;
    1. list all the differences between the data we used for training and the data they used. 
1. You probably have spotted in question 1.A. that we did not implement the *"validation stopping"* that the authors mention in Section 2.2.1 of their article. To solve this, create a new function named `train_pytorch_net_early_stopping` that trains a neural network but also implements the stopping procedure they describe. **Hint**: You can start by copying the code of `train_pytorch_net`.
1. Make a new pipeline that implements all the difference you spotted and try to reproduce the plot from the article's Figure 7.
1. Did you manage to reproduce the plot? If not, suggest some hypotheses to explain why.


**Answers:**
> Answer here


In [ ]:
# Or here:

---

## (optional) Exercise 3: Cross-validation on multiple subjects
In this exercise, we would like to compare EEGNet on all the subjects of the BNCI2014001 dataset. 
For this, we will use a library called [skorch](https://skorch.readthedocs.io/en/stable/index.html) that allows you to wrap any pytorch module into a scikit-learn classifier. 
This way, you can use your pytorch module in any scikit-learn pipeline and with every scikit-learn evaluation function!

**Question:** 
1. Look at the example we provided below, which wraps our EEGNet module into a sklearn classifier with skorch. Use the documentation of [`skorch.NeuralNetClassifier`](https://skorch.readthedocs.io/en/stable/classifier.html#skorch.classifier.NeuralNetClassifier) to make sure you understand how to use this class.
1. Use the `moabb.evaluation.WithinSessionEvaluation` class from assignment 12 to evaluate a wrapped EEGNet (with the hyper-parameters you want) on all the subjects of the BNCI2014001 dataset.
1. Re-use the plot functions we defined in assignment 12 to visualise your new EEGNet results along side those you obtained last week with the 'FBCSP' and 'CSP[8-12]' pipelines in exercise 2.A.


**NOTE**:
The training of the model across multiple subjects will take you quite some time (about 12h on an M1 Pro). Consider running this overnight if you want to compute the full evaluation, or reduce the number of training epochs :)

In [ ]:
from skorch import NeuralNetClassifier

net = NeuralNetClassifier(
    module=EEGNetv4,
    module__in_chans=X.shape[1], 
    module__n_classes=n_classes, 
    module__input_window_samples=X.shape[2],
    module__drop_prob=0.5,
    max_epochs=5,
    batch_size=50,
    device='cpu',
    optimizer=torch.optim.Adam,
    lr=0.001, # default learning rate for Adam
    # train_split=None,
    verbose=1, # use verbose=0 to remove the summary at every epoch
)
le = LabelEncoder()
net = net.fit(X, le.fit_transform(y))


---

## Exercise 0: Who did what?
Please provide a short description on who contributed what to your submission.

> Answer here